### Pré-traitement

In [26]:
# Import des librairies
import pandas as pd
import numpy as np
import hashlib
from rapidfuzz import process, fuzz
import re

In [27]:
# On récupère les données de stats
data_ids = pd.read_csv("../data/entity_resolution.csv")
data_opta_analyst = pd.read_csv("../data/silver_analyst.csv")
data_fotmob = pd.read_csv("../data/fotmob_v2.csv")
data_sofascore = pd.read_csv("../data/sofascore_v2.csv")
data_understat = pd.read_csv("../data/silver_understat.csv")


In [28]:
# On crée l'identifiant issu d'opta analyst
def create_player_id(analyst_name):
    return hashlib.sha256(str(analyst_name).encode("utf-8")).hexdigest()[:16]

data_ids["opta_id"] = data_ids["analyst_name"].apply(create_player_id)

In [29]:
data_ids

,analyst_name,best_tm_id,best_method,fuzzy_score,reep_id,fotmob_id,sofascore_id,fbref_id,understat_id,opta_id
0,Aaron Greene,122407.0,reep,NaN,reep_pf7029e95,NaN,NaN,NaN,NaN,41e1a6833619b240
1,Aarón Escandell,284430.0,reep,NaN,reep_pbc6fe046,534955.0,368922.0,67669ce7,NaN,be5c61f30246a2b9
2,Aarón Herrera,401362.0,reep,NaN,reep_pcdb31119,NaN,NaN,d86e3070,NaN,bf8da63c2c534563
3,Aarón Martín,251878.0,reep,NaN,reep_pe0bb5f2e,684981.0,797286.0,2f3e911a,NaN,17b08b01696a4b12
4,Abakar Sylla,962555.0,reep,NaN,reep_peeaa665d,1359613.0,1170197.0,NaN,NaN,1b43e3b148b564e9
...,...,...,...,...,...,...,...,...,...,...
8636,Zachary Athekame,990637.0,reep,NaN,reep_p5406864a,1595629.0,1409700.0,NaN,NaN,2b627ddf55fdb08b
8637,Zander Clark,98067.0,reep,NaN,reep_p2b342b28,NaN,556366.0,d7fc839a,NaN,fce47f1f77818a44
8638,Óscar Trejo,30321.0,reep,NaN,reep_pab0700e3,21414.0,21949.0,fc647b34,NaN,5bd027a5f1bf4062
8639,Óscar Valentín,517753.0,reep,NaN,reep_p9d45ac50,956622.0,900008.0,592f3158,NaN,9486ca987c85e01a


In [30]:
# On comptabilise le nombre d'identifiants disponible pour un joueur sur les différents fournisseurs de données

cols_ids = ["best_tm_id","fotmob_id","sofascore_id","opta_id"]

n_complet = data_ids[cols_ids].notna().all(axis=1).sum()

print(n_complet)

n_total = len(data_ids)
pct_complet = n_complet / n_total * 100

print(f"{n_complet} lignes sur {n_total} ({pct_complet:.1f} %)")

1957
1957 lignes sur 8641 (22.6 %)


### Opta

In [31]:
# On enlève les joueurs ayant participé à moins de 90 minute dans une compétition
data_opta_analyst = data_opta_analyst[data_opta_analyst["minutes"] >= 90].copy()

# On garde que les minutes jouées dans le big 5 + compétitions européennes
top_leagues = ["Premier League","Serie A","La Liga","Ligue 1","Bundesliga","Champions League","Europa League","Conference League"]

# On enlève les joueurs problématiques au niveau des homonymes
data_opta_analyst = data_opta_analyst[
    ~data_opta_analyst["name"].isin(["Nico González","Vitinha","Weysley"])
].copy()

data_opta_analyst = data_opta_analyst[
    data_opta_analyst["league"].isin(top_leagues)
].copy()
# On ajoute l'identifiant d'opta
data_opta_analyst = data_opta_analyst.merge(data_ids[["analyst_name", "opta_id"]],
    left_on="name",right_on="analyst_name",how="left")

# Suppression de la colonne analyst_name ajoutée par le merge
data_opta_analyst = data_opta_analyst.drop(columns="analyst_name")

# On garde uniquement les lignes ayant opta_id
data_opta_analyst = data_opta_analyst[data_opta_analyst["opta_id"].notna()].copy()

# Variables à sommer
sum_cols = ["apps","minutes","atk_goals","atk_xg","atk_goals_vs_xg","atk_shots","atk_shots_on_target","def_tackles",
    "def_interceptions","def_possession_won","def_blocks","def_clearances","def_ground_duels_total",
    "def_ground_duels_won","def_aerial_duels_total","def_aerial_duels_won","pass_total","pass_open_play_total",
    "pass_final_third","pass_crosses","pass_long_total","pass_through_balls","carry_all_carries","carry_progressive",
    "carry_distance_m","carry_prog_distance_m","carry_lead_to_shot","carry_lead_to_goal","carry_lead_to_chance","carry_lead_to_assist",
    "gk_goals_conceded","gk_saves","gk_xgot_conceded","gk_goals_prevented"]

# Variables à moyenner en fonction des minutes
weighted_cols = ["atk_conversion_pct","atk_xg_per_shot","def_ground_duels_pct","def_aerial_duels_pct",
    "pass_accuracy_pct","pass_open_play_pct","pass_long_pct","carry_avg_distance_m","carry_prog_avg_distance_m","gk_save_pct"]


# Variables pour lesquelles on créera un per90
per90_cols = ["atk_goals","atk_xg","atk_shots","atk_shots_on_target","def_tackles","def_interceptions","def_possession_won",
    "def_blocks","def_clearances","def_ground_duels_total","def_ground_duels_won","def_aerial_duels_total",
    "def_aerial_duels_won","pass_total","pass_open_play_total","pass_final_third","pass_crosses","pass_long_total",
    "pass_through_balls","carry_all_carries","carry_progressive","carry_distance_m","carry_prog_distance_m",
    "carry_lead_to_shot","carry_lead_to_goal","carry_lead_to_chance","carry_lead_to_assist",
    "gk_goals_conceded","gk_saves","gk_xgot_conceded","gk_goals_prevented"]


# On ne garde que les colonnes réellement présentes
sum_cols = [
    col for col in sum_cols
    if col in data_opta_analyst.columns
]

weighted_cols = [
    col for col in weighted_cols
    if col in data_opta_analyst.columns
]

per90_cols = [
    col for col in per90_cols
    if col in data_opta_analyst.columns
]

def aggregate_player_opta(group):

    result = {}

    # group.name = (opta_id, season)
    opta_id, season = group.name

    result["opta_id"] = opta_id
    result["season"] = season

    # League : concatène uniquement les valeurs différentes
    result["league"] = ", ".join(
        group["league"]
        .dropna()
        .astype(str)
        .drop_duplicates()
    )

    # Sommes
    for col in sum_cols:
        result[col] = group[col].sum(min_count=1)

    # Moyennes pondérées par les minutes
    for col in weighted_cols:

        mask = (group[col].notna() & group["minutes"].notna() & (group["minutes"] > 0))

        if mask.any():
            result[col] = np.average(
                group.loc[mask, col],
                weights=group.loc[mask, "minutes"]
            )
        else:
            result[col] = np.nan

    # Autres variables : première valeur disponible
    processed_cols = (
        {"opta_id", "season", "league"}
        | set(sum_cols)
        | set(weighted_cols)
    )

    other_cols = [
        col for col in group.columns
        if col not in processed_cols
    ]

    for col in other_cols:

        values = group[col].dropna()

        result[col] = (
            values.iloc[0]
            if len(values) > 0
            else np.nan
        )

    return pd.Series(result)


# Agrégation par joueur ET par saison
data_opta_analyst_agg = (
    data_opta_analyst
    .groupby(
        ["opta_id", "season"],
        dropna=False
    )
    .apply(aggregate_player_opta)
    .reset_index(drop=True)
)


# Création des variables /90
for col in per90_cols:

    data_opta_analyst_agg[f"{col}_per90"] = np.where(data_opta_analyst_agg["minutes"] > 0,
        data_opta_analyst_agg[col] / data_opta_analyst_agg["minutes"] * 90, np.nan)


# Mettre name en première colonne
cols = ["name"] + [
    col for col in data_opta_analyst_agg.columns
    if col != "name"
]

data_opta_analyst_agg = data_opta_analyst_agg[cols]

# On enlève les joueurs ayant participé à moins de 270 minutes en cummulé sur plusieurs compétitions sur une saison
data_opta_analyst_agg = data_opta_analyst_agg[data_opta_analyst_agg["minutes"] >= 270].copy()

# Liste des big_5 leagues
big_5_leagues = ["Premier League","Serie A","La Liga","Ligue 1","Bundesliga"]

data_complete = data_opta_analyst_agg[
    data_opta_analyst_agg["league"].apply(
        lambda x: any(league in x.split(", ") for league in big_5_leagues)
    )
].copy()

In [32]:
data_complete

,name,opta_id,season,league,apps,minutes,atk_goals,atk_xg,atk_goals_vs_xg,atk_shots,...,carry_distance_m_per90,carry_prog_distance_m_per90,carry_lead_to_shot_per90,carry_lead_to_goal_per90,carry_lead_to_chance_per90,carry_lead_to_assist_per90,gk_goals_conceded_per90,gk_saves_per90,gk_xgot_conceded_per90,gk_goals_prevented_per90
0,Gianluigi Donnarumma,000b408e096a7ad7,2025/2026,"Premier League, Champions League",43,3870,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.953488,2.302326,1.111628,0.158140
1,Gianluigi Donnarumma,000b408e096a7ad7,2026/2027,"Premier League, Champions League",10,900,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.300000,2.400000,1.480000,0.180000
2,Lorenzo Pellegrini,000c9c4ad406cee3,2025/2026,"Serie A, Europa League",33,1988,7.0,6.96,0.04,47.0,...,116.827968,45.235412,0.181087,0.0,0.316901,0.000000,NaN,NaN,NaN,NaN
6,Álvaro Valles,00235a13fa36141b,2025/2026,"La Liga, Europa League",33,2970,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.151515,2.909091,1.093939,-0.027273
7,Álvaro Valles,00235a13fa36141b,2026/2027,"La Liga, Europa League",6,540,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.333333,1.833333,0.450000,0.116667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6242,Ebenezer Akinsanmiro,ff7b5978731bb67d,2025/2026,Serie A,24,1445,0.0,0.86,-0.86,13.0,...,121.901730,61.106574,0.186851,0.0,0.186851,0.062284,NaN,NaN,NaN,NaN
6243,Matteo Politano,ff9588c500ef382c,2025/2026,"Serie A, Champions League",40,2546,2.0,2.60,-0.60,52.0,...,214.412804,118.912412,0.742341,0.0,0.671642,0.070699,NaN,NaN,NaN,NaN
6249,Natan,ffa4e1255914e995,2025/2026,"Europa League, La Liga",42,3575,0.0,0.53,-0.53,11.0,...,125.607273,59.911049,0.000000,0.0,0.025175,0.000000,NaN,NaN,NaN,NaN
6250,Natan,ffa4e1255914e995,2026/2027,"La Liga, Europa League",10,855,0.0,0.03,-0.03,1.0,...,165.789474,87.842105,0.000000,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN


In [33]:
#data_opta_analyst_agg.to_csv("data_opta_analyst_agg.csv",index=False)

### Sofascore

In [34]:
# On ajoute l'identifiant de fotmob afin d'associer ces données à celle d'opta

data_complete = data_complete.merge(data_ids[["opta_id", "sofascore_id"]],on="opta_id",how="left")

data_complete["sofascore_id"] = data_complete["sofascore_id"].fillna("")

In [35]:
# On analyse le nombre de lignes où sofascore_id est manquant
print((data_complete["sofascore_id"] == "").sum())

157


In [36]:
# On récupère les joueurs sans sofascore_id

missing_sofa = data_complete[data_complete["sofascore_id"].isna() | (data_complete["sofascore_id"] == "")].copy()

# Liste des noms disponibles dans data_sofascore
sofa_names = data_sofascore["name"].dropna().unique().tolist()

# Fonction de fuzzy matching

def get_sofascore_match(player_name):

    if pd.isna(player_name):
        return pd.Series([None, None, None])

    match = process.extractOne(player_name,sofa_names,scorer=fuzz.ratio)

    if match is None:
        return pd.Series([None, None, None])

    matched_name, score, _ = match

    # On conserve uniquement les correspondances >= 85 %
    if score >= 85:

        player_id = data_sofascore.loc[data_sofascore["name"] == matched_name,"player_id"].iloc[0]

        return pd.Series([player_id,matched_name,score])

    return pd.Series([None,None,score])

# Fuzzy matching

missing_sofa[["sofascore_id_fuzzy", "sofascore_name_match", "match_score"]] = missing_sofa["name"].apply(get_sofascore_match)

# On ajoute les sofascore_id trouvés dans data_complete

data_complete.loc[missing_sofa.index,"sofascore_id"] = missing_sofa["sofascore_id_fuzzy"]

# Correspondances manuelles pour les joueurs restants
manual_sofascore_ids = {
    "Mohamed Meïté": 1606642,
    "Beraldo": 1108441,
    "Milan Djuric": 76132,
    "Copete": 913695,
    "Al Musrati": 868958,
    "Lee Jae-Sung": 537552,
    "CJ Egan-Riley" :1131448,
    "Lionel Mpasi": 599192,
    "Alex Amorim" : 1511706,
    "Djené Dakonam" : 307702, 
    "Hákon Haraldsson":1138804,
    "Ibrahim Sulemana":105905,
    "Djordje Petrovic":882604,
    "Étienne Youté":980406
    }

# Ajout des sofascore_id manuels
mask_manual = data_complete["name"].isin(manual_sofascore_ids)

data_complete.loc[mask_manual,"sofascore_id"] = data_complete.loc[mask_manual,"name"].map(manual_sofascore_ids)

# Nombre de lignes restantes sans sofascore_id

mask_missing_final = (data_complete["sofascore_id"].isna() |(data_complete["sofascore_id"] == ""))

print("Nombre de lignes restantes sans sofascore_id :",mask_missing_final.sum())

# Affichage des joueurs restant sans sofascore_id

data_complete.loc[mask_missing_final,["name", "opta_id"]]

# On enlève les lignes sans sofascore_id
data_complete = data_complete[data_complete["sofascore_id"].notna() &(data_complete["sofascore_id"] != "")].copy()

# Réinitialisation de l'index
data_complete = data_complete.reset_index(drop=True)

print("Nombre de lignes restantes dans data_complete :", len(data_complete))

Nombre de lignes restantes sans sofascore_id : 7
Nombre de lignes restantes dans data_complete : 2324


In [37]:
# On fait une copie de data_sofascore
data_sofascore_work = data_sofascore.copy()

# Prétraitement de la saison
def extract_season(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    match = re.search(r"(\d{2})/(\d{2})", value)

    if match:
        start = match.group(1)
        end = match.group(2)

        return f"20{start}/20{end}"

    match_year = re.search(r"\b(20\d{2})\b", value)

    if match_year:
        return match_year.group(1)

    return np.nan

data_sofascore_work["season"] = data_sofascore_work["season"].apply(extract_season)

# Uniformisation des identifiants
data_complete["sofascore_id"] = data_complete["sofascore_id"].astype(str).str.replace(".0", "", regex=False).str.strip()

data_sofascore_work["player_id"] = data_sofascore_work["player_id"].astype(str).str.replace(".0", "", regex=False).str.strip()

# Variables à sommer
sum_cols_sofascore = ["stat_bigChancesCreated","stat_bigChancesMissed","stat_keyPasses","stat_penaltiesTaken","stat_offsides"]

# Variables déjà calculées par 90 minutes
weighted_cols_sofascore = ["stat_km_per90","stat_sprints_per90"]

# Variable où l'on garde la valeur maximale
max_cols_sofascore = ["stat_topSpeed"]

# Conversion en numérique
all_stats_sofascore = (sum_cols_sofascore+ weighted_cols_sofascore+ max_cols_sofascore)

for col in all_stats_sofascore:
    data_sofascore_work[col] = pd.to_numeric(data_sofascore_work[col],errors="coerce")

# Conversion des minutes SofaScore
data_sofascore_work["stat_minutesPlayed"] = pd.to_numeric(data_sofascore_work["stat_minutesPlayed"],errors="coerce")

# Somme des statistiques de volume par joueur et saison
data_sofascore_sum = data_sofascore_work.groupby(["player_id","season"],as_index=False)[sum_cols_sofascore].sum(min_count=1)

# Fonction de moyenne pondérée par les minutes SofaScore
def weighted_average(group,col):
    valid = (group[col].notna() & group["stat_minutesPlayed"].notna() & (group["stat_minutesPlayed"] > 0))

    if valid.sum() == 0:
        return np.nan

    return np.average(group.loc[valid,col],weights=group.loc[valid,"stat_minutesPlayed"])

# Moyenne pondérée des statistiques déjà en per90
weighted_results = []

for (player_id,season),group in data_sofascore_work.groupby(["player_id","season"]):
    row = {"player_id": player_id,"season": season}

    for col in weighted_cols_sofascore:
        row[col] = weighted_average(group,col)

    weighted_results.append(row)

data_sofascore_weighted = pd.DataFrame(weighted_results)

# Top speed : maximum par joueur et saison
data_sofascore_max = data_sofascore_work.groupby(["player_id","season"],as_index=False)[max_cols_sofascore].max()

# Regroupement de toutes les statistiques
data_sofascore_agg = data_sofascore_sum.merge(data_sofascore_weighted,on=["player_id","season"],how="outer").merge(
    data_sofascore_max,on=["player_id","season"],how="outer")

# Correction manuelle de certains sofascore_id
manual_sofascore_ids = {
    "Abdoulaye Faye": "1529051",
    "Álex Moreno": "294593",
    "Álvaro García": "345111",
    "Ameen Al Dakhil": "1979970",
    "Amine Sbaï": "1392829",
    "André Almeida": "845693",
    "André Silva": "190159",
    "Angel Gomes": "867441",
    "Antonín Kinsky": "1031251",
    "Boubacar Traoré": "982613",
    "Bruno Fernandes": "288205",
    "Callum Wilson": "113956",
    "Daniel Svensson": "1021272",
    "Endrick": "1174937",
    "Estêvão": "1597265",
    "Ibrahim Sangaré": "843754",
    "Ismaël Koné": "1134351",
    "Javi Guerra": "1122610",
    "Jesús Rodríguez": "1800245",
    "Johan Vásquez": "889785",
    "Juan Cruz": "814360",
    "Juan Rodríguez": "1485391",
    "Karim Coulibaly": "1797852",
    "Lewis Cook": "548188",
    "Luis Díaz": "883537",
    "Luiz Felipe": "850035",
    "Malick Fofana": "1195784",
    "Mamadou Coulibaly": "1410148",
    "Marcus Pedersen": "934409",
    "Mohamed Bamba": "1405239",
    "Ousmane Camara": "1049538",
    "Pablo Ibáñez": "1084381",
    "Pablo Martínez": "927066",
    "Rodrigo Muniz": "1015256",
    "Rodrigo Ribeiro": "1215904",
    "Rodrigo Riquelme": "989113",
    "Sergio Herrera": "294377",
    "Thomas Kristensen": "1063373"}

# On remplace le sofascore_id uniquement pour les joueurs renseignés manuellement
mask_manual_sofascore = data_complete["name"].isin(manual_sofascore_ids)

data_complete.loc[mask_manual_sofascore,"sofascore_id"] = data_complete.loc[mask_manual_sofascore,"name"].map(manual_sofascore_ids)

# Jointure avec data_complete
data_complete = data_complete.merge(data_sofascore_agg,left_on=["sofascore_id","season"],right_on=["player_id","season"],how="left")

# Suppression de player_id ajouté par la jointure
data_complete = data_complete.drop(columns="player_id")

# Conversion des minutes Opta présentes dans data_complete
data_complete["minutes"] = pd.to_numeric(data_complete["minutes"],errors="coerce")

# Calcul des statistiques par 90 minutes à partir des minutes de data_complete
data_complete["stat_bigChancesCreated_per90"] = np.where(data_complete["minutes"] > 0,
    data_complete["stat_bigChancesCreated"] / data_complete["minutes"] * 90,np.nan)

data_complete["stat_bigChancesMissed_per90"] = np.where(data_complete["minutes"] > 0,
    data_complete["stat_bigChancesMissed"] / data_complete["minutes"] * 90,np.nan)

data_complete["stat_keyPasses_per90"] = np.where(data_complete["minutes"] > 0,
    data_complete["stat_keyPasses"] / data_complete["minutes"] * 90,np.nan)

# Liste des statistiques SofaScore finales
sofascore_cols_final = ["stat_bigChancesCreated","stat_bigChancesCreated_per90","stat_bigChancesMissed","stat_bigChancesMissed_per90",
    "stat_keyPasses","stat_keyPasses_per90","stat_penaltiesTaken","stat_offsides","stat_km_per90","stat_sprints_per90","stat_topSpeed"]

# Vérifications finales
print("\nNombre de lignes dans data_complete :",len(data_complete))

# Lignes sans aucune statistique SofaScore
mask_no_sofascore_stats = data_complete[sofascore_cols_final].isna().all(axis=1)

print("\nNombre de lignes sans aucune statistique SofaScore :",mask_no_sofascore_stats.sum())

# On enlève les lignes sans aucune statistique SofaScore
data_complete = data_complete[~mask_no_sofascore_stats].copy()

# Réinitialisation de l'index
data_complete = data_complete.reset_index(drop=True)

print("Nombre de lignes restantes dans data_complete :",len(data_complete))


Nombre de lignes dans data_complete : 2324

Nombre de lignes sans aucune statistique SofaScore : 182
Nombre de lignes restantes dans data_complete : 2142


In [38]:
# On récupère les lignes sans aucune statistique SofaScore
#data_sans_sofascore_stats = data_complete[mask_no_sofascore_stats].copy()

# Export en CSV
#data_sans_sofascore_stats.to_csv("data_sans_sofascore_stats.csv",index=False,encoding="utf-8-sig")


In [114]:
#data_complete.to_csv("data_complete.csv",index=False)

### Fotmob